# Week5_ex5 - Eddy Current Braking, Materials & Speed Sweep [Full License]

Full (non-Student) license variant of `Week5_ex5.ipynb`, which itself builds on `Week5_ex4.ipynb`'s eddy-current braking setup. Same materials/speed sweep -- a bar magnet swept across a range of speeds over copper and brass plates, geometry rebuilt and re-solved per combination, solid loss converted to an approximate braking force, quadratic curve fit at the end -- but this version assumes a full Ansys license so the Transient solver actually runs. Kept as a separate file so the Student-blocked version stays as-is for the portfolio narrative, and this one is ready to go whenever a full license is available. Path handling and project setup follow the same portable pattern used throughout this repo.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
# 1. Launch AEDT (once, before the sweep loop) -- assumes a full (non-Student) Ansys license, so student_version is hardcoded False
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=False)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "Transient"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week5_ex5_full_license"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print("Project directory:", dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week5_ex5_full_license"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
# Magnet material, set up once and reused across every run in the sweep
magnet_mat = M3D.materials.duplicate_material("NdFe35", name="magnet_n")
magnet_mat.get_magnetic_coercivity()
magnet_mat.set_magnetic_coercivity(value=-890000, x=0, y=0, z=1)

In [ ]:
## Sweep parameters ##
# The original notebook collected these five values through input() prompts.
# Hardcoded here instead so the notebook runs top-to-bottom without any typing --
# the exact numbers don't matter much since this can't be run locally anyway.

speed_start = 0.02   # m/sec
speed_end = 0.10     # m/sec
speed_step = 0.02    # m/sec
common_speeds = np.arange(speed_start, speed_end + 0.0001, speed_step)

Mesh1_length = 2   # Magnet mesh max length (mm)
Mesh2_length = 5   # Metal mesh max length (mm)
Mesh3_length = 5   # Band mesh max length (mm)

time_step_val = 100   # ms
time_step = f"{time_step_val}ms"

# List of (material, speed) combinations to run
simulation_runs = [("copper", s) for s in common_speeds] + [("brass", s) for s in common_speeds]

# Accumulates one summary dict per simulation run
all_results = []

# Base plate/magnet/band dimensions (mm), used as plain Python multipliers below
X_dim = 10
Y_dim = 10
Z_dim = 10
O = 0

In [ ]:
## Run one simulation per (material, speed) combination ##

for i, (material, speed) in enumerate(simulation_runs):
    print(f"\n==== Run {i+1}: Material = {material}, Speed = {speed} m/sec ====")

    # Region and geometry
    region = M3D.modeler.create_region(pad_value=[500, 500, 500, 500, 500, 500], pad_type='Absolute Offset', name='Region')

    # Metal plate
    origin = [O, O, O]
    sizes = [20*X_dim, 15*Y_dim, Z_dim]
    Metal = M3D.modeler.create_box(origin, sizes, name="Metal", material=material)

    # Bar magnet
    origin = [8*X_dim, O, Z_dim]
    sizes = [2*X_dim, 4*X_dim, Z_dim*0.5]
    magnet = M3D.modeler.create_box(origin, sizes, name="Magnet", material="magnet_n")
    M3D.modeler.move(magnet, [X_dim, O, 0.01*Z_dim])

    # Band wrapping the moving magnet
    origin = [X_dim*-0.1, Y_dim*-0.1, Z_dim+0.05]
    sizes = [4*X_dim+2, 15*Y_dim+2, Z_dim+0.01]
    band = M3D.modeler.create_box(origin, sizes, name="Band", material="Vacuum")
    M3D.modeler.move(band, [8*X_dim, O, O])

    # Transient setup
    setup = M3D.create_setup(name='MySetupAuto', setup_type="Transient")
    setup.props["StopTime"] = "1s"
    setup.props["TimeStep"] = time_step
    setup.props["SaveFieldsType"] = "Every N Steps"
    setup.props["N Steps"] = "1s"
    setup.props["Steps From"] = "0s"
    setup.props["Steps To"] = "1s"

    # Mesh
    meshing = M3D.mesh.assign_length_mesh(["Magnet"], inside_selection=True, maximum_length=Mesh1_length, maximum_elements=None, name="Mesh1")
    meshing = M3D.mesh.assign_length_mesh(["Metal"], inside_selection=True, maximum_length=Mesh2_length, maximum_elements=None, name="Mesh2")
    meshing = M3D.mesh.assign_length_mesh(["Band"], inside_selection=True, maximum_length=Mesh3_length, maximum_elements=None, name="Mesh3")

    # Motion
    velocity_string = f"{speed}m_per_sec"
    print(f"Run {i+1} -- velocity: {velocity_string}")

    Motionsetup = M3D.assign_translate_motion(
        "Band", coordinate_system='Global', axis='Y',
        positive_movement=True, start_position=0, periodic_translate=False,
        negative_limit=0, positive_limit=110, velocity=velocity_string,
        mechanical_transient=False, mass=0, damping=0, load_force=0,
        motion_name="MotionSetup1"
    )

    # Eddy effects on the plate
    effect = M3D.eddy_effects_on(Metal, enable_eddy_effects=True, enable_displacement_current=True)

    # Solve
    setup.analyze()

    # Report + CSV export
    report_module = oDesign.GetModule("ReportSetup")
    report_module.CreateReport(
        "Loss Plot", "Transient", "Rectangular Plot", "MySetupAuto : Transient",
        ["Domain:=", "Sweep"],
        ["Time:=", ["All"]],
        ["X Component:=", "Time", "Y Component:=", ["SolidLoss"]]
    )

    csv_filename = os.path.join(dir, f"loss_plot_{i+1}.csv")
    report_module.ExportToFile("Loss Plot", csv_filename, False)

    # Read the exported data back in
    df = pd.read_csv(csv_filename)
    x_label = df.columns[0]
    y_label = df.columns[1]

    # Normalize the loss column to watts (AEDT may export mW or uW depending on scale)
    if "mW" in y_label:
        df.iloc[:, 1] = df.iloc[:, 1] / 1000.0
        new_label = y_label.replace("mW", "W")
        df.rename(columns={y_label: new_label}, inplace=True)
        y_label = new_label
    elif "uW" in y_label:
        df.iloc[:, 1] = df.iloc[:, 1] / 1000000.0
        new_label = y_label.replace("uW", "W")
        df.rename(columns={y_label: new_label}, inplace=True)
        y_label = new_label

    # Loss vs time for this run
    plt.figure()
    plt.plot(df.iloc[:, 0], df.iloc[:, 1], label=y_label, color=("red" if material == "copper" else "blue"))
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(f"Solid Loss vs Time (Run {i+1}) - {material}, {velocity_string}")
    plt.grid(True)
    plt.legend()
    plt.show()

    solid_loss = df.iloc[:, 1]
    current_mean = solid_loss.mean()

    all_results.append({
        "Simulation": i+1,
        "Material": material,
        "Speed(m/sec)": speed,
        "Mean_SolidLoss(W)": current_mean,
        "Max_SolidLoss(W)": solid_loss.max(),
        "Min_SolidLoss(W)": solid_loss.min()
    })

    # Tear down this run's geometry/setup/mesh so the next iteration starts clean
    try:
        oModule = oDesign.GetModule("ModelSetup")
        oModule.DeleteMotionSetup(["MotionSetup1"])
    except:
        pass
    oModule = oDesign.GetModule("MeshSetup")
    for mesh_name in ["Mesh1", "Mesh2", "Mesh3"]:
        try:
            oModule.DeleteOp([mesh_name])
        except:
            pass
    M3D.delete_setup("MySetupAuto")
    M3D.modeler.delete(assignment="Metal")
    M3D.modeler.delete(assignment="Magnet")
    M3D.modeler.delete(assignment="Band")

In [ ]:
## Save the combined summary and plot speed vs. approximate force ##

results_df = pd.DataFrame(all_results)
# saved flat, next to the notebook -- same convention as Images/
excel_path = "Week5_ex5_combined_results.xlsx"
results_df.to_excel(excel_path, index=False)
print(f"All run summaries saved to {excel_path}")

# Force ~ Power / Speed -- a simple approximation, not a rigorous force calculation
copper_speeds, copper_forces = [], []
brass_speeds, brass_forces = [], []

for res in all_results:
    mat = res["Material"]
    spd = res["Speed(m/sec)"]
    loss = res["Max_SolidLoss(W)"]

    if spd != 0:
        force = loss / spd
        if mat == "copper":
            copper_speeds.append(spd)
            copper_forces.append(force)
        elif mat == "brass":
            brass_speeds.append(spd)
            brass_forces.append(force)

plt.figure()
plt.plot(copper_speeds, copper_forces, color="red", marker="o", label="Copper")
plt.plot(brass_speeds, brass_forces, color="blue", marker="^", label="Brass")
plt.xlabel("Speed (m/sec)")
plt.ylabel("Force (N)")
plt.title("Speed vs. Force (Force = Max_SolidLoss / Speed)")
plt.grid(True)
plt.legend()
plt.show()

print("Sweep finished.")

In [ ]:
## Quadratic curve fit for each material ##

def poly2(x, a, b, c):
    return a * x**2 + b * x + c

c_x = np.array(copper_speeds)
c_y = np.array(copper_forces)
b_x = np.array(brass_speeds)
b_y = np.array(brass_forces)

popt_copper, _ = curve_fit(poly2, c_x, c_y)
popt_brass, _ = curve_fit(poly2, b_x, b_y)

c_y_fit = poly2(c_x, *popt_copper)
b_y_fit = poly2(b_x, *popt_brass)

print(f"Copper fit: {popt_copper[0]:.4f}x^2 + {popt_copper[1]:.4f}x + {popt_copper[2]:.4f}")
print(f"Brass  fit: {popt_brass[0]:.4f}x^2 + {popt_brass[1]:.4f}x + {popt_brass[2]:.4f}")

os.makedirs("Images", exist_ok=True)

plt.figure()
plt.plot(copper_speeds, copper_forces, "ro", label="Copper raw")
plt.plot(c_x, c_y_fit, "r--", label="Copper fit (2nd order)")

plt.plot(brass_speeds, brass_forces, "b^", label="Brass raw")
plt.plot(b_x, b_y_fit, "b--", label="Brass fit (2nd order)")

plt.xlabel("Speed (m/sec)")
plt.ylabel("Force (N)")
plt.title("Speed vs. Force with Curve Fitting")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("Images/Week5_ex5_speed_vs_force_fit.png", dpi=150)
plt.show()

In [ ]:
## Save project ##

M3D.save_project()